# Hyperparameter Tuning & GridSearchCV

Welcome, This course covers the fundamentals of hyperparameter tuning, the mechanics of Cross-Validation, and practical implementation using `scikit-learn`'s `GridSearchCV`.

## Module 1: Introduction to Hyperparameters
* **1.1 Parameters vs. Hyperparameters:** Understanding the difference between model-learned parameters (e.g., weights in linear regression) and user-defined hyperparameters (e.g., learning rate, tree depth).
* **1.2 The Need for Tuning:** How hyperparameters affect model performance (Underfitting vs. Overfitting).
* **1.3 The Curse of Dimensionality in Tuning:** Why exhaustive search becomes computationally expensive.

# The Core Definition

## Model Parameters
These are the internal variables that the model **learns automatically** from the training data.  
The data scientist does **not** set these manually.  

They are the **output of the learning process**.

Examples:
- Weights in Linear Regression
- Coefficients in Logistic Regression
- Weights and biases in Neural Networks
- Split thresholds in Decision Trees


---

## Hyperparameters
These are the **external configuration settings** that the data scientist sets **before the training process begins**.  

They guide **how the model learns**.

Examples:
- Learning rate
- Number of trees in Random Forest
- Maximum depth of a tree
- Number of hidden layers in a Neural Network
- Regularization strength (λ)


---

## Key Difference

| Model Parameters | Hyperparameters |
|------------------|------------------|
| Learned automatically | Set manually before training |
| Internal to the model | External configuration |
| Output of training | Input to training |
| Example: weights | Example: learning rate |

# Examples: Parameters vs Hyperparameters

---

## 1️⃣ Linear Regression

### Parameters:
- The weights ($w_1, w_2, \dots, w_n$)
- The bias ($b$)

The model calculates these automatically during training  
(e.g., using **Ordinary Least Squares**).

### Hyperparameters:
- If using Gradient Descent → the **Learning Rate ($\alpha$)**  

You must set this **before training begins**.

---

## 2️⃣ Decision Trees

### Parameters:
- The actual decision rules at each node  
  (e.g., *"Is Age > 25?"*, *"Is Income < $50k?"*)

The model determines these splits based on the data.

### Hyperparameters:
- **Maximum Depth (`max_depth`)**

You decide beforehand how deep the tree can grow  
(e.g., maximum 5 levels).

---

![Underfitting vs Good Fit vs Overfitting](figure.jpg)

# Understanding Underfitting, Good Fit, and Overfitting

---

## 📉 Left Side – Underfitting (bad on both training and testing)

- **Model is too simple**
- **High bias**
- **Poor performance** on training and testing data
- **Example:** Linear model trying to fit complex data


---

## ✅ Middle – Good Fit (Optimal Model)

- **Balanced bias and variance**
- **Learns patterns properly**
- **Performs well** on unseen data
- **Best generalization**

💡 *This is the goal of training.*

---

## 📈 Right Side – Overfitting  (great on training, bad on testing)

- **Model is too complex**
- **Memorizes training data**
- **High variance**
- **Poor performance** on new data
- **Example:** Deep neural network on small dataset


---

# 1.3 The Curse of Dimensionality in Hyperparameter Tuning

---

## 🎯 The Goal
To understand **why we can't just try every possible hyperparameter** and why tuning can take a lot of time.

---

## 📖 The Core Concept
**The Curse of Dimensionality** means:  
> As you add more hyperparameters to tune, the number of combinations grows **exponentially**.  

Checking every combination (brute force) **quickly becomes impossible**.

---

## 💡 Analogy: The Combination Padlock
- **1 dial (0-9):** 10 tries max  
- **3 dials:** 10 × 10 × 10 = 1,000 tries  
- **5 dials:** 10 × 10 × 10 × 10 × 10 = 100,000 tries  

> Every new dial (hyperparameter) **multiplies the work**.

---

## 🛠️ Mathematical Example

Suppose we tune a **Random Forest Classifier** with these options:

| Hyperparameter | Options |
|----------------|---------|
| n_estimators | 10, 50, 100, 200, 500 (5 options) |
| max_depth | None, 10, 20, 30 (4 options) |
| min_samples_split | 2, 5, 10, 15, 20 (5 options) |


**Total combinations:**  

5 × 4 × 5  = 100 unique models


Using **5-Fold Cross-Validation**:  


100 combinations × 5 folds = 500 model trainings



If **1 model takes 1 minute**, the total time = **8.33 hours**!  
Add one more hyperparameter with 10 options → **83.3 hours (~3, 4 days)**!

---


# Module 2: Cross-Validation (The Foundation)

---

## **2.1 What is Cross-Validation?**  
Cross-validation is a technique to **evaluate how well your model will perform on unseen data**.  

- The simple **train-test split** divides your dataset once into a training set and a test set.  
- Problem: The model’s performance depends heavily on **which data ends up in the training vs test set**.  
- **Cross-validation** solves this by **splitting the data multiple times** and averaging results for a more reliable estimate.  

💡 *Analogy:* Testing a recipe multiple times on different groups of friends instead of relying on just one tasting session.  

---

## **2.2 K-Fold Cross-Validation**  
K-Fold Cross-Validation splits the dataset into **K equal parts (folds)**:

1. Use **K-1 folds** to train the model.  
2. Use the **remaining fold** to test the model.  
3. Repeat this process **K times**, each time using a different fold as the test set.  
4. Average the performance across all K runs → **robust estimate of model performance**.  

- Common choice: **K = 5 or 10**  
- Pros: More reliable than a single train-test split, especially on small datasets  

💡 *Example:* With 5-Fold CV, each fold is tested once → 5 performance scores → take the average.  

---

## **2.3 Stratified K-Fold**  
- Sometimes datasets are **imbalanced** (e.g., 90% class A, 10% class B).  
- Regular K-Fold may split data unevenly → some folds might have very few minority class samples.  
- **Stratified K-Fold** ensures **each fold preserves the class proportion** of the original dataset.  

💡 *Analogy:* Think of making sure each tasting group gets a **balanced mix of cookie types** instead of ending up with only chocolate chip cookies in one group.  

- In scikit-learn:

```python
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5)
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

# Module 3: Introduction to GridSearchCV

---

## **3.1 The Grid Search Concept**

**The Goal:**  
You need to **visualize what a "Grid" means** in hyperparameter tuning. We are **systematically defining a finite set of values** to test.

---

### 📖 Core Concept
Grid Search is an **exhaustive search algorithm**:

- You provide a list of hyperparameters and specific values to test.  
- It builds a **mathematical grid** of all possible combinations.  
- Each combination is evaluated to find the **best hyperparameters**.

---

### 💡 Analogy: Restaurant Combo Meal
Imagine building a combo meal:

| Choice | Options | Number of options |
|--------|--------|-----------------|
| Main   | Burger, Chicken Sandwich, Veggie Wrap | 3 |
| Side   | Fries, Salad | 2 |
| Drink  | Soda, Water | 2 |

- Total meals to try = 3 × 2 × 2 = **12 meals**  
- A **Grid Search** is like trying **every possible combo** to find the best one.

---

### 🛠️ Technical Example (Random Forest)

```python
param_grid = {
    'n_estimators': [10, 50],       # 2 options
    'max_depth': [5, 10, None],     # 3 options
}

## 3.2 How GridSearchCV Works (Grid Search + K-Fold CV)

**The Goal:**  
Must understand that **each combination is evaluated rigorously using Cross-Validation**.

---

### 📖 Core Concept
**GridSearchCV = Grid Search + Cross-Validation**  

- For **each hyperparameter combination**, it runs a **full K-Fold Cross-Validation**.  
- The **best parameters** are the ones with the **highest average score** across all folds.

---

### 💡 Analogy: Taste Test Panel
- One person tasting all meals → **biased opinion**.  
- 5 judges tasting all meals → **average score is fairer**.  
- Similarly, **5-Fold CV evaluates each combination on multiple folds** to ensure reliability.

---

### 🛠️ Step-by-Step
1. Pick **combination #1** from the grid.  
2. Split the training data into **K folds**.  
3. Train on **K-1 folds**, test on the remaining fold.  
4. Repeat **K times** → calculate **average score** for combination #1.  
5. Repeat steps 1–4 for **all combinations**.  
6. Compare all averages → pick the **best combination**.

---

In [1]:

import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.metrics import roc_auc_score

In [2]:
# Load dataset
digits = load_digits()
digits

{'data': array([[ 0.,  0.,  5., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ..., 10.,  0.,  0.],
        [ 0.,  0.,  0., ..., 16.,  9.,  0.],
        ...,
        [ 0.,  0.,  1., ...,  6.,  0.,  0.],
        [ 0.,  0.,  2., ..., 12.,  0.,  0.],
        [ 0.,  0., 10., ..., 12.,  1.,  0.]]),
 'target': array([0, 1, 2, ..., 8, 9, 8]),
 'frame': None,
 'feature_names': ['pixel_0_0',
  'pixel_0_1',
  'pixel_0_2',
  'pixel_0_3',
  'pixel_0_4',
  'pixel_0_5',
  'pixel_0_6',
  'pixel_0_7',
  'pixel_1_0',
  'pixel_1_1',
  'pixel_1_2',
  'pixel_1_3',
  'pixel_1_4',
  'pixel_1_5',
  'pixel_1_6',
  'pixel_1_7',
  'pixel_2_0',
  'pixel_2_1',
  'pixel_2_2',
  'pixel_2_3',
  'pixel_2_4',
  'pixel_2_5',
  'pixel_2_6',
  'pixel_2_7',
  'pixel_3_0',
  'pixel_3_1',
  'pixel_3_2',
  'pixel_3_3',
  'pixel_3_4',
  'pixel_3_5',
  'pixel_3_6',
  'pixel_3_7',
  'pixel_4_0',
  'pixel_4_1',
  'pixel_4_2',
  'pixel_4_3',
  'pixel_4_4',
  'pixel_4_5',
  'pixel_4_6',
  'pixel_4_7',
  'pixel_5_0',
  'pixel_5_1',
 

In [3]:
# Convert to DataFrame
df = pd.DataFrame(data=digits.data, columns=digits.feature_names)

In [4]:
df

,pixel_0_0,pixel_0_1,pixel_0_2,pixel_0_3,pixel_0_4,pixel_0_5,pixel_0_6,pixel_0_7,pixel_1_0,pixel_1_1,...,pixel_6_6,pixel_6_7,pixel_7_0,pixel_7_1,pixel_7_2,pixel_7_3,pixel_7_4,pixel_7_5,pixel_7_6,pixel_7_7
0,0.0,0.0,5.0,13.0,9.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,6.0,13.0,10.0,0.0,0.0,0.0
1,0.0,0.0,0.0,12.0,13.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,11.0,16.0,10.0,0.0,0.0
2,0.0,0.0,0.0,4.0,15.0,12.0,0.0,0.0,0.0,0.0,...,5.0,0.0,0.0,0.0,0.0,3.0,11.0,16.0,9.0,0.0
3,0.0,0.0,7.0,15.0,13.0,1.0,0.0,0.0,0.0,8.0,...,9.0,0.0,0.0,0.0,7.0,13.0,13.0,9.0,0.0,0.0
4,0.0,0.0,0.0,1.0,11.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,16.0,4.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1792,0.0,0.0,4.0,10.0,13.0,6.0,0.0,0.0,0.0,1.0,...,4.0,0.0,0.0,0.0,2.0,14.0,15.0,9.0,0.0,0.0
1793,0.0,0.0,6.0,16.0,13.0,11.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,6.0,16.0,14.0,6.0,0.0,0.0
1794,0.0,0.0,1.0,11.0,15.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,9.0,13.0,6.0,0.0,0.0
1795,0.0,0.0,2.0,10.0,7.0,0.0,0.0,0.0,0.0,0.0,...,2.0,0.0,0.0,0.0,5.0,12.0,16.0,12.0,0.0,0.0


In [5]:
digits.data

array([[ 0.,  0.,  5., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ..., 10.,  0.,  0.],
       [ 0.,  0.,  0., ..., 16.,  9.,  0.],
       ...,
       [ 0.,  0.,  1., ...,  6.,  0.,  0.],
       [ 0.,  0.,  2., ..., 12.,  0.,  0.],
       [ 0.,  0., 10., ..., 12.,  1.,  0.]])

In [6]:
digits.target


array([0, 1, 2, ..., 8, 9, 8])

In [7]:
X, y = digits.data, digits.target

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define the hyperparameter grid
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5, 10],
    'criterion':['gini','entropy']
}

# Initialize RandomForestClassifier
rf = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=7,              # 5-Fold Cross Validation
    n_jobs=-1,         # Use all CPU cores
    verbose=2          # Prints progress
)

# Fit the Grid Search to your training data
grid_search.fit(X_train, y_train)

# Best combination found
print("Best Hyperparameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)

Fitting 7 folds for each of 36 candidates, totalling 252 fits
Best Hyperparameters: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}
Best Cross-Validation Score: 0.9784208923920029


In [9]:
# Predict on the test set
y_pred = grid_search.predict(X_test)
y_pred_proba = grid_search.predict_proba(X_test)

# Evaluate the model
roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print("ROC AUC Score on test data: ", roc_auc)
print(y_pred_proba[36])

ROC AUC Score on test data:  0.9992930613967651
[0.06111111 0.11909091 0.01090909 0.01       0.17777778 0.08
 0.08       0.10111111 0.32875    0.03125   ]


In [10]:
y_test[36]

6

In [11]:
print(y_test)

[5 2 8 1 7 2 6 2 6 5 0 5 9 3 4 4 2 4 9 9 6 3 8 1 2 5 6 0 3 4 6 7 2 6 6 6 6
 5 0 9 1 7 9 6 5 7 5 2 7 5 0 8 5 5 3 2 4 0 0 2 7 5 6 1 3 7 6 5 7 0 9 0 3 8
 2 5 8 2 3 5 9 3 2 7 9 6 8 1 0 1 2 9 4 9 1 2 7 8 4 2 6 9 3 7 3 9 6 1 1 0 9
 2 1 6 3 4 8 7 1 0 0 4 6 5 8 2 8 8 3 0 0 8 6 4 3 9 3 3 3 3 0 7 0 0 1 9 5 8
 1 5 0 6 6 6 6 1 7 7 6 7 7 8 7 3 6 5 9 0 3 8 0 9 8 1 9 5 9 5 8 9 9 7 9 1 9
 5 4 7 3 0 4 9 7 7 5 6 5 8 3 4 5 4 9 2 5 5 2 1 3 8 8 9 3 6 1 0 1 4 0 5 5 6
 6 7 4 3 8 4 1 0 7 9 2 8 4 8 4 2 4 0 0 0 2 6 7 0 4 5 2 2 9 0 4 6 8 2 3 9 2
 3 0 6 8 7 1 4 4 1 1 6 3 8 1 2 5 7 8 3 2 0 3 4 1 9 9 9 6 3 7 1 6 9 4 7 1 8
 1 3 0 5 3 4 1 9 3 5 4 7 4 1 5 1 5 0 9 8 4 2 3 8 4 1 2 0 1 1 4 4 5 7 5 0 3
 2 2 4 2 7 7 8 7 6 3 1 1 5 8 8 8 6 7 2 7 8 9 4 2 0 3 4]


In [12]:
print(y_pred)

[5 2 8 1 7 2 6 2 6 5 0 5 9 3 4 4 2 4 9 9 6 3 8 1 2 5 6 0 3 4 6 7 2 6 6 6 8
 5 0 9 1 7 9 6 5 7 5 2 7 5 0 1 5 5 3 2 4 0 0 2 7 5 6 1 3 7 6 5 7 0 9 0 3 8
 2 5 8 2 3 5 9 3 2 7 9 6 8 1 0 1 2 9 4 8 1 2 7 8 4 2 6 9 3 7 3 9 6 1 1 0 9
 2 1 6 3 4 8 7 1 0 0 4 6 5 8 2 8 1 3 0 0 8 6 4 3 9 3 3 3 3 4 7 0 0 1 9 5 8
 1 5 0 6 6 6 6 1 7 7 6 7 7 8 7 3 6 5 9 0 3 8 0 9 8 1 9 5 9 5 8 9 9 7 3 1 9
 5 4 7 3 0 4 9 7 7 5 6 5 8 3 4 5 4 9 2 5 5 2 1 3 8 8 9 3 6 1 0 1 4 0 5 5 6
 6 7 4 3 8 4 5 0 7 9 2 1 4 8 4 2 4 0 0 0 2 6 7 0 4 5 2 2 9 0 4 6 8 2 3 9 2
 3 0 6 8 7 1 4 4 1 1 6 3 8 1 2 5 7 7 3 2 0 3 4 1 9 9 9 6 3 7 1 6 9 4 7 1 8
 1 3 0 5 3 4 1 9 3 5 4 7 4 1 5 1 5 0 9 8 1 2 3 8 4 1 2 0 1 1 4 4 5 7 5 0 3
 2 2 4 2 7 7 8 7 6 3 1 1 5 8 8 8 6 7 2 7 8 9 4 2 0 3 4]


In [13]:
for i, j in zip(y_test, y_pred):
    print(i,j)

5 5
2 2
8 8
1 1
7 7
2 2
6 6
2 2
6 6
5 5
0 0
5 5
9 9
3 3
4 4
4 4
2 2
4 4
9 9
9 9
6 6
3 3
8 8
1 1
2 2
5 5
6 6
0 0
3 3
4 4
6 6
7 7
2 2
6 6
6 6
6 6
6 8
5 5
0 0
9 9
1 1
7 7
9 9
6 6
5 5
7 7
5 5
2 2
7 7
5 5
0 0
8 1
5 5
5 5
3 3
2 2
4 4
0 0
0 0
2 2
7 7
5 5
6 6
1 1
3 3
7 7
6 6
5 5
7 7
0 0
9 9
0 0
3 3
8 8
2 2
5 5
8 8
2 2
3 3
5 5
9 9
3 3
2 2
7 7
9 9
6 6
8 8
1 1
0 0
1 1
2 2
9 9
4 4
9 8
1 1
2 2
7 7
8 8
4 4
2 2
6 6
9 9
3 3
7 7
3 3
9 9
6 6
1 1
1 1
0 0
9 9
2 2
1 1
6 6
3 3
4 4
8 8
7 7
1 1
0 0
0 0
4 4
6 6
5 5
8 8
2 2
8 8
8 1
3 3
0 0
0 0
8 8
6 6
4 4
3 3
9 9
3 3
3 3
3 3
3 3
0 4
7 7
0 0
0 0
1 1
9 9
5 5
8 8
1 1
5 5
0 0
6 6
6 6
6 6
6 6
1 1
7 7
7 7
6 6
7 7
7 7
8 8
7 7
3 3
6 6
5 5
9 9
0 0
3 3
8 8
0 0
9 9
8 8
1 1
9 9
5 5
9 9
5 5
8 8
9 9
9 9
7 7
9 3
1 1
9 9
5 5
4 4
7 7
3 3
0 0
4 4
9 9
7 7
7 7
5 5
6 6
5 5
8 8
3 3
4 4
5 5
4 4
9 9
2 2
5 5
5 5
2 2
1 1
3 3
8 8
8 8
9 9
3 3
6 6
1 1
0 0
1 1
4 4
0 0
5 5
5 5
6 6
6 6
7 7
4 4
3 3
8 8
4 4
1 5
0 0
7 7
9 9
2 2
8 1
4 4
8 8
4 4
2 2
4 4
0 0
0 0
0 0
2 2
6 6
7 7
0 0
4 4
5 5
2 2
2 2


In [14]:
# Module 3: Using GridSearchCV with Breast Cancer Dataset

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

data = load_breast_cancer()
X = data.data
y = data.target

In [15]:
# Convert to DataFrame
df = pd.DataFrame(data=data.data, columns=data.feature_names)
df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [16]:
data.data

array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
        1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
        8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
        8.758e-02],
       ...,
       [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
        7.820e-02],
       [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
        1.240e-01],
       [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
        7.039e-02]])

In [17]:
data.target

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0,

In [18]:
#  Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#  Define the Random Forest model
rf = RandomForestClassifier(random_state=42)

# Set up hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

In [ ]:
# Initialize RandomForestClassifier
rf = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=7,              # 5-Fold Cross Validation
    n_jobs=-1,         # Use all CPU cores
    verbose=2          # Prints progress
)

# Fit the Grid Search to your training data
grid_search.fit(X_train, y_train)

# Best combination found
print("Best Hyperparameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)

Fitting 7 folds for each of 162 candidates, totalling 1134 fits


In [ ]:
# Predict on the test set
y_pred = grid_search.predict(X_test)
y_pred_proba = grid_search.predict_proba(X_test)

# Evaluate the model
roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print("ROC AUC Score on test data: ", roc_auc)
print(y_pred_proba[0])

In [ ]:
y_pred1 = grid_search.predict(new_patient1)
print(y_pred1)

In [ ]:
new_patient3 = np.array([[12.0, 18.0, 75.0, 450.0, 0.09, 0.15, 0.8, 12.0, 0.015, 0.02,
     0.25, 0.8, 1.8, 25.0, 0.005, 0.015, 0.025, 0.01, 0.015, 0.002,
     14.0, 22.0, 85.0, 500.0, 0.12, 0.28, 1.0, 16.0, 0.025, 0.03]

])
